In [1]:
!nvidia-smi

%cd /content
!rm -rf CIRI-FS
!git clone --branch asal/CiriEXT4 https://github.com/isusbu/CIRI-FS.git
%cd /content/CIRI-FS

!git branch --show-current

Wed Jul 22 17:12:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q -r requirements.txt
!pip install -q transformers accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.6 MB/s eta 0:00:00


In [3]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [4]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
!find "/content/drive/MyDrive" -maxdepth 6 \
    \( -iname "*cpd*" -o -iname "*ocd*" -o -iname "*ccd*" -o -iname "*ext4*" \) \
    | sort

/content/drive/MyDrive/CIRI_EXT4
/content/drive/MyDrive/Colab Notebooks/Copy of qwen_ext4_cpd_zero_shot
/content/drive/MyDrive/Colab Notebooks/EXT4SD
/content/drive/MyDrive/Colab Notebooks/qwen_ext4_cpd_zero_shot


In [6]:
!ls -lah "/content/drive/MyDrive/CIRI_EXT4"

ls: /content/drive/MyDrive/CIRI_EXT4/Dataset: No such file or directory
total 4.5K
lrw------- 1 root root    0 Jul 22 03:59 Dataset -> /content/drive/.shortcut-targets-by-id/1xrerJFYBKPY28WmDN6o8zkkE73mSAHn0/Dataset
drwx------ 3 root root 4.0K Jul 13 05:28 Qwen2.5-Coder-7B-Instruct
-rw------- 1 root root  212 Jul 13 05:30 Qwen_zero_shot_metrics.txt


In [7]:
!readlink -f "/content/drive/MyDrive/CIRI_EXT4/Dataset"

/content/drive/.shortcut-targets-by-id/1xrerJFYBKPY28WmDN6o8zkkE73mSAHn0/Dataset


In [8]:
import os

DRIVE_DATA = os.path.realpath(
    "/content/drive/MyDrive/CIRI_EXT4/Dataset"
)

print(DRIVE_DATA)
print("Exists:", os.path.exists(DRIVE_DATA))

/content/drive/.shortcut-targets-by-id/1xrerJFYBKPY28WmDN6o8zkkE73mSAHn0/Dataset
Exists: True


In [9]:
DRIVE_DATA = "/content/drive/.shortcut-targets-by-id/1xrerJFYBKPY28WmDN6o8zkkE73mSAHn0/Dataset/EXT4_XML_Dataset"

In [10]:
!ls icse25_data/datasets/synthesize_config

alluxio  etcd	  ground_truth	hcommon  postgresql  yarn
django	 ext4_SD  hbase		hdfs	 redis	     zookeeper


In [11]:
!cp -r "$DRIVE_DATA/ext4_CCD" \
    icse25_data/datasets/synthesize_config/

!cp "$DRIVE_DATA/Groundtruth/ext4_CCD.tsv" \
    icse25_data/datasets/synthesize_config/ground_truth/

In [12]:
!find icse25_data/datasets/synthesize_config/ext4_CCD/correct -type f | wc -l
!find icse25_data/datasets/synthesize_config/ext4_CCD/erroneous -type f | wc -l

57
57


In [13]:
!ls icse25_data/datasets/synthesize_config/ground_truth

alluxio.tsv  ext4_CCD.tsv  hcommon.tsv	   redis.tsv
django.tsv   ext4_SD.tsv   hdfs.tsv	   yarn.tsv
etcd.tsv     hbase.tsv	   postgresql.tsv  zookeeper.tsv


In [15]:
!python -m py_compile \
    ciri/ciri_eng.py \
    ciri/ciri_runner.py \
    ciri/query/llm_gen.py

In [16]:
!grep -n "\[Qwen Cost\]" ciri/query/llm_gen.py

262:            "[Qwen Cost] "


In [17]:
BENCHMARK = "ext4_CCD"
MODEL = "Qwen2.5-Coder-7B-Instruct"
MODE = "zero_shot"

OUTPUT_ROOT = (
    f"icse25_data/results/synthesize_config/"
    f"{BENCHMARK}/{MODEL}/{MODE}"
)

In [18]:
!rm -rf "$OUTPUT_ROOT"

In [19]:
!python -m ciri.ciri_eng \
  --input_path "icse25_data/datasets/synthesize_config/$BENCHMARK/erroneous" \
  --output_path "$OUTPUT_ROOT/erroneous" \
  --model "$MODEL" \
  --system ext4 \
  --version 1.47.0 \
  --validconfig_shot_num 0 \
  --misconfig_shot_num 0 \
  --file_format xml

2026-07-22 17:21:14 - Ciri - INFO - Using device: CUDA
2026-07-22 17:21:14 - Ciri - INFO - Using dtype: torch.bfloat16
config.json: 100% 663/663 [00:00<00:00, 3.38MB/s]
model.safetensors.index.json: 100% 27.8k/27.8k [00:00<00:00, 87.6MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0% 0/4 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/6.02G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/15.2G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/15.2G [00:00<?, ?B/s]
Reconstructing (incomplete total...):  11% 1.65G/15.2G [00:22<05:54, 38.4MB/s, 14.2MB/s  ]
Reconstructing (incomplete total...):  11% 1.68G/15.2G [00:22<05:15, 42.9MB/s, 19.9MB/s  ]
Reconstructing (incomplete total...):  13% 1.92G/15.2G [00:40<17:31, 12.7MB/s, 9.83MB/s  ]
Reconstructing (incomplete total...):  13% 1.92G/15.2G [00:40<17:31, 12.7MB/s, 8.16MB/s  ]
Reconstructing (incomplete total...):  15% 2.25G/15.2

In [20]:
!python -m ciri.ciri_eng \
  --input_path "icse25_data/datasets/synthesize_config/$BENCHMARK/correct" \
  --output_path "$OUTPUT_ROOT/correct" \
  --model "$MODEL" \
  --system ext4 \
  --version 1.47.0 \
  --validconfig_shot_num 0 \
  --misconfig_shot_num 0 \
  --file_format xml

2026-07-22 17:53:59 - Ciri - INFO - Using device: CUDA
2026-07-22 17:53:59 - Ciri - INFO - Using dtype: torch.bfloat16
Loading weights:   1% 2/339 [00:08<24:31,  4.37s/it]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100% 339/339 [01:00<00:00,  5.58it/s]
2026-07-22 17:55:08 - Ciri - INFO - Model loaded successfully on CUDA!
2026-07-22 17:55:08 - Ciri - INFO - [llm_gen] Using device: CUDA
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
2026-07-22 17:55:18 - Ciri - INFO - [Qwen Cost] call=1, input_tokens=456, output_tokens=96, total_tokens=552, generation_time_seconds=

In [21]:
!echo "Correct results:"
!find "$OUTPUT_ROOT/correct" -type f | wc -l

!echo "Erroneous results:"
!find "$OUTPUT_ROOT/erroneous" -type f | wc -l

Correct results:
57
Erroneous results:
57


In [22]:
!python icse25_data/script/result_parser.py \
  --project "$BENCHMARK" \
  --model "$MODEL" \
  --mode "$MODE"

Error parsing result in /content/CIRI-FS/icse25_data/script/../results/synthesize_config/ext4_CCD/Qwen2.5-Coder-7B-Instruct/zero_shot/erroneous/58: [Errno 2] No such file or directory: '/content/CIRI-FS/icse25_data/script/../results/synthesize_config/ext4_CCD/Qwen2.5-Coder-7B-Instruct/zero_shot/erroneous/58'
[Ciri Result] on ext4_CCD with Qwen2.5-Coder-7B-Instruct and zero_shot mode
File-Level: Precision: 0.90, Recall: 0.98, Accuracy: 0.94, F1: 0.94
Param-Level: Precision: 0.00, Recall: 0.00, Accuracy: 0.86, F1: N.A.


In [23]:
from pathlib import Path

base = Path(
    "icse25_data/results/synthesize_config/"
    "ext4_CCD/Qwen2.5-Coder-7B-Instruct/zero_shot"
)

for category in ["correct", "erroneous"]:
    existing = sorted(
        int(p.name)
        for p in (base / category).iterdir()
        if p.name.isdigit()
    )

    missing_1_to_57 = [
        i for i in range(1, 58)
        if i not in existing
    ]

    print(category)
    print("Number of results:", len(existing))
    print("Missing from 1–57:", missing_1_to_57)

correct
Number of results: 57
Missing from 1–57: []
erroneous
Number of results: 57
Missing from 1–57: []
